# core 交互探索笔记本：亲手算一次 Hartree–Fock

欢迎！这个 notebook 带你**一步步运行**教学量子化学程序 `core`，
亲眼看到分子、基组、积分矩阵和 SCF 迭代的每个中间量。

配套阅读材料：[docs/TUTORIAL_zh.md](docs/TUTORIAL_zh.md)（理论推导与公式）。

**使用方法**：从上到下依次运行每个单元格（`Shift+Enter`）。
遇到 🔍 **试一试** 的提示时，修改代码再运行，观察结果变化——这是本 notebook 的正确打开方式。

**环境要求**：`numpy`、`scipy`、`rdkit`、`matplotlib`（绘图用）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from core import Molecule, rhf, build_basis

np.set_printoptions(precision=4, suppress=True)   # 矩阵打印更好看
print("环境就绪！")

## 1. 构建第一个分子：H$_2$

量子化学使用**原子单位制**：长度单位是 Bohr（1 Å = 1.8897 Bohr），能量单位是 Hartree（1 Ha = 27.211 eV = 627.5 kcal/mol）。

先手动搭一个氢分子，核间距取教科书上的 1.4 Bohr：

In [ ]:
h2 = Molecule.from_atoms(
    [("H", (0.0, 0.0, 0.0)),
     ("H", (0.0, 0.0, 1.4))],
    unit="bohr", name="H2")

print(h2)
print(f"\n电子数: {h2.n_electrons}")
print(f"核排斥能 E_nn = {h2.nuclear_repulsion():.6f} Ha")
print("验证: Z1*Z2/R = 1*1/1.4 =", 1 / 1.4)

### 用 SMILES 自动生成三维结构

手动输入坐标只适合双原子分子。对于更大的分子，我们让 **RDKit** 从 SMILES 字符串自动生成三维几何构型（距离几何嵌入 + MMFF94 力场优化）：

| SMILES | 分子 |
|---|---|
| `O` | 水 |
| `C` | 甲烷 |
| `N` | 氨 |
| `F` | 氟化氢 |
| `CO` | 甲醇 |

🔍 **试一试**：把下面的 `"O"` 换成表格里其他 SMILES，观察原子数和电子数的变化。

In [ ]:
water = Molecule.from_smiles("O", name="水")
print(water)
print(f"\n电子数: {water.n_electrons}（O 有 8 个 + 2 个 H 各 1 个）")

## 2. 基组：每个原子带来哪些基函数？

分子轨道被展开为**基函数**（原子轨道）的线性组合。STO-3G 是最小基组：

- H、He：只有 1 个 `1s` 函数；
- Li–F：`1s` + `2s` + 三个 `2p`，共 5 个函数。

每个基函数都是 3 个高斯函数的固定线性组合（"收缩"）：

In [ ]:
basis = build_basis(water)
print(f"水分子共有 {len(basis)} 个基函数：\n")
for bf in basis:
    print(f"  {bf.label:10s}  角动量(l,m,n)={bf.lmn}  "
          f"高斯指数={np.round(bf.exps, 3)}")

画出氢原子 1s 基函数的"收缩"过程——3 个高斯函数如何拼出一个近似的 Slater 轨道：

In [ ]:
r = np.linspace(0, 5, 300)
h_1s = build_basis(h2)[0]                    # H 的 1s 基函数（已归一化）

total = np.zeros_like(r)
for a, c in zip(h_1s.exps, h_1s.coefs):
    g = c * np.exp(-a * r**2)                # coefs 里已含每个原始函数的归一化
    plt.plot(r, g, "--", label=f"高斯 α={a:.3f}")
    total += g
plt.plot(r, total, "b-", lw=2.5, label="三者之和 (STO-3G)")
plt.plot(r, np.exp(-r) / np.sqrt(np.pi), "k-", alpha=0.5, lw=2.5,
         label="精确 Slater 轨道")
plt.xlabel("r / Bohr"); plt.ylabel("振幅"); plt.legend()
plt.title("STO-3G: 3 Gaussians imitate 1 Slater orbital")
plt.show()

🔍 **试一试**：注意核处（r=0）——高斯函数是圆头的，真实轨道有尖点（cusp）。这就是最小基组误差的来源之一。

## 3. 积分矩阵：SCF 之前的"一次性"工作

HF 需要 4 组积分（详见教程第 4 节）。对 H$_2$（只有 2 个基函数），矩阵小到可以直接看：

In [ ]:
from core.integrals import build_one_electron, build_eri

basis_h2 = build_basis(h2)
S, T, V = build_one_electron(basis_h2, h2)

print("重叠矩阵 S（对角元=1：基函数已归一化；非对角元=两个 1s 的重叠）")
print(S)
print("\n动能矩阵 T")
print(T)
print("\n核吸引矩阵 V（注意全是负的——吸引作用）")
print(V)

🔍 **试一试**：把第 1 节里 H$_2$ 的核间距从 1.4 改成 3.0 Bohr，重新运行到这里。
S 的非对角元（两个原子轨道的重叠）会变大还是变小？为什么？

双电子积分是一个 4 维张量 $(\mu\nu\vert\lambda\sigma)$：

In [ ]:
eri = build_eri(basis_h2)
print("ERI 张量形状:", eri.shape)
print(f"(11|11) = {eri[0,0,0,0]:.6f}   同一轨道内两电子的排斥")
print(f"(11|22) = {eri[0,0,1,1]:.6f}   不同原子上两团电子云的排斥")
print(f"(12|12) = {eri[0,1,0,1]:.6f}   交换积分")

## 4. 运行第一个完整的 HF 计算

把水分子交给 `rhf()`，观察 SCF 迭代的全过程。注意能量**单调下降**并逐渐收敛：

In [ ]:
result = rhf(water)

`rhf()` 返回一个字典，包含所有你可能想分析的量：

In [ ]:
print("result 的键:", list(result.keys()))

HARTREE_TO_EV = 27.211386
eps = result["mo_energies"]
n_occ = water.n_electrons // 2

print(f"\n总能量          = {result['energy']:.6f} Ha")
print(f"HOMO 能量       = {eps[n_occ-1]:.4f} Ha")
print(f"LUMO 能量       = {eps[n_occ]:.4f} Ha")
print(f"HOMO-LUMO 能隙  = {(eps[n_occ]-eps[n_occ-1])*HARTREE_TO_EV:.2f} eV")
print(f"\nKoopmans 定理预测电离能 ≈ -E(HOMO) = "
      f"{-eps[n_occ-1]*HARTREE_TO_EV:.1f} eV（实验值 12.6 eV）")
print(f"\nMulliken 电荷: {np.round(result['mulliken_charges'], 3)}")
print("（O 带负电、H 带正电——水的极性从第一性原理算出来了！）")

分子轨道系数矩阵 $C$ 的每一列是一条 MO。看看 HOMO 由哪些原子轨道组成：

In [ ]:
C = result["mo_coefficients"]
labels = [bf.label for bf in result["basis"]]

print("HOMO 的组成（系数绝对值越大贡献越大）：")
for lab, c in zip(labels, C[:, n_occ - 1]):
    bar = "#" * int(abs(c) * 20)
    print(f"  {lab:10s} {c:+.4f}  {bar}")

🔍 **试一试**：HOMO 应该几乎是纯的 O 2p 轨道（垂直于分子平面的孤对电子）。
把 `n_occ - 1` 改成其他列号，看看更深的成键轨道里 O 和 H 是如何混合的。

## 5. 势能曲线：拉断一个 H$_2$ 分子

对一系列核间距逐点算能量，就得到势能曲线——化学键的"模样"：

In [ ]:
distances = np.linspace(0.8, 5.0, 25)          # 单位 Bohr
energies = []
for d in distances:
    mol = Molecule.from_atoms([("H", (0, 0, 0)), ("H", (0, 0, d))],
                              unit="bohr")
    energies.append(rhf(mol, verbose=False)["energy"])
energies = np.array(energies)

i_min = energies.argmin()
print(f"平衡键长 ≈ {distances[i_min]:.2f} Bohr（实验值 1.40 Bohr）")
print(f"最低能量 = {energies[i_min]:.5f} Ha")

plt.plot(distances, energies, "o-")
plt.axhline(-1.0, ls=":", color="gray",
            label="2 isolated H atoms, exact (-1 Ha)")
plt.xlabel("H-H distance / Bohr"); plt.ylabel("RHF energy / Ha")
plt.legend(); plt.title("H$_2$ dissociation curve")
plt.show()

**思考题**：距离拉大时曲线为什么停在远高于 $-1$ Ha 的地方？
（提示：限制性 HF 强迫两个电子始终待在同一条空间轨道里，见教程第 7 节——这是引入"电子相关"方法的经典动机。）

---

## 6. 练习 A：水的键角扫描

下面的代码扫描 H–O–H 键角并画出能量曲线。运行它，然后回答：

1. 最优键角是多少？和实验值 104.5° 差多少？
2. 为什么 90° 和 120° 的能量都更高？

In [ ]:
r_oh = 0.9578                                   # O–H 键长 / Angstrom
angles = np.linspace(85, 125, 9)                # 键角 / 度

e_angle = []
for theta in angles:
    half = np.radians(theta) / 2
    mol = Molecule.from_atoms(
        [("O", (0.0, 0.0, 0.0)),
         ("H", (0.0,  r_oh * np.sin(half), -r_oh * np.cos(half))),
         ("H", (0.0, -r_oh * np.sin(half), -r_oh * np.cos(half)))],
        unit="angstrom")
    e_angle.append(rhf(mol, verbose=False)["energy"])

best = angles[np.argmin(e_angle)]
print(f"能量最低的键角 ≈ {best:.0f}°（实验值 104.5°）")

plt.plot(angles, e_angle, "o-")
plt.axvline(104.5, ls=":", color="gray", label="experimental angle")
plt.xlabel("H-O-H angle / deg"); plt.ylabel("RHF energy / Ha")
plt.legend(); plt.title("Water bond-angle scan")
plt.show()

🔍 **进一步探索**：把扫描点加密（如 30 个点），或同时扫描键长 `r_oh`，
做一张二维势能面图（`plt.contourf`）。

## 7. 练习 B：Koopmans 定理趋势

对氢化物系列计算 $-\varepsilon_{HOMO}$，与实验电离能对比：

In [ ]:
series = {"CH4": "C", "NH3": "N", "H2O": "O", "HF": "F"}
exp_ie = {"CH4": 12.6, "NH3": 10.1, "H2O": 12.6, "HF": 16.1}   # 实验值/eV

print(f"{'分子':>5} {'-E(HOMO)/eV':>12} {'实验电离能/eV':>12}")
for name, smiles in series.items():
    mol = Molecule.from_smiles(smiles, name=name)
    res = rhf(mol, verbose=False)
    homo = res["mo_energies"][mol.n_electrons // 2 - 1]
    print(f"{name:>5} {-homo * 27.211386:12.1f} {exp_ie[name]:12.1f}")

**思考题**：最小基组 STO-3G 算出的数值系统性偏低，但**趋势**对不对？
哪个分子偏差最大？猜猜为什么。

## 8. 练习 C：你的回合——HeH$^+$

最简单的异核体系（Szabo & Ostlund 书中的主角之一），文献值 $E = -2.860$ Ha。

**任务**：仿照第 1 节，用 `Molecule.from_atoms` 构建 HeH$^+$
（He 在原点，H 在 z 轴 1.4632 Bohr 处，**注意 charge=1**），
然后运行 `rhf` 并与文献值比较。

In [ ]:
# === 在这里写你的代码 ===
# 提示:
# hehp = Molecule.from_atoms([...], charge=1, unit="bohr", name="HeH+")
# result = rhf(hehp)


## 9. 更多挑战（选做）

1. **变分原理体验**：修改 `core/basis.py` 中氢的某个 STO-3G 指数（±20%），
   重算 H$_2$。能量只会升高——想清楚为什么。（改完记得重启内核！）
2. **计算量标度**：用 `%time` 给 H$_2$O、NH$_3$、CH$_4$ 的 `rhf` 计时，
   验证积分计算的 $\mathcal{O}(K^4)$ 标度。
3. **DIIS 加速**（进阶）：阅读 `core/scf.py`，用误差矩阵
   $e = FDS - SDF$ 实现 Pulay 的 DIIS，把水的迭代次数从 16 减到 ~8。
4. **偶极矩**（进阶）：在 `integrals.py` 里补充偶极积分
   $\langle\mu\vert x\vert\nu\rangle$（提示：只需 Hermite 系数 $E_t^{ij}$ 的 $t=0,1$ 项），
   算出水的偶极矩并与实验值 1.85 Debye 比较。

---

**恭喜完成！** 你已经运行过一个从积分到 SCF 全部透明可见的量子化学程序。
接下来可以阅读 [docs/TUTORIAL_zh.md](docs/TUTORIAL_zh.md) 补全每一步背后的推导。